## 1. Load data

In [2]:
import pandas as pd

registered = pd.read_csv("../data/registered_bike_rentals.csv")
direct = pd.read_csv("../data/direct_pickup_bike_rentals.csv")
weather = pd.read_csv("../data/weather.csv")
holidays = pd.read_csv("../data/holidays.csv")

## 2. Data observation
### 2.1 check rental data

In [3]:
display(registered.head())
print("Registered shape:", registered.shape)
print("Registered columns:", registered.columns.tolist())

,id,datetime,user_id,location_id
0,1,2011-01-01 00:05:09,158,16
1,2,2011-01-01 00:05:21,262,18
2,3,2011-01-01 00:05:39,68,18
3,4,2011-01-01 00:12:05,12,9
4,5,2011-01-01 00:25:58,91,11


Registered shape: (2672662, 4)
Registered columns: ['id', 'datetime', 'user_id', 'location_id']


In [4]:
display(direct.head())
print("Direct shape:", direct.shape)
print("Direct columns:", direct.columns.tolist())

,id,datetime,user_id,location_id
0,1,2011-01-01 00:24:04,232,2
1,2,2011-01-01 00:30:19,54,14
2,3,2011-01-01 00:39:08,201,5
3,4,2011-01-01 01:01:12,298,13
4,5,2011-01-01 01:02:37,23,14


Direct shape: (620017, 4)
Direct columns: ['id', 'datetime', 'user_id', 'location_id']


registered 和 direct 都是 rental records。The colunms of registered and direct are same, so they can be merged directly.这说明之后这两个表可以用相同逻辑处理。 

In [7]:
registered.columns.equals(direct.columns)

True

### 2.2 check datetime

In [8]:
registered.info()

<class 'pandas.DataFrame'>
RangeIndex: 2672662 entries, 0 to 2672661
Data columns (total 4 columns):
 #   Column       Dtype
---  ------       -----
 0   id           int64
 1   datetime     str  
 2   user_id      int64
 3   location_id  int64
dtypes: int64(3), str(1)
memory usage: 81.6 MB


In [9]:
direct.info()

<class 'pandas.DataFrame'>
RangeIndex: 620017 entries, 0 to 620016
Data columns (total 4 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   id           620017 non-null  int64
 1   datetime     620017 non-null  str  
 2   user_id      620017 non-null  int64
 3   location_id  620017 non-null  int64
dtypes: int64(3), str(1)
memory usage: 18.9 MB


确认datetime数据是按秒的时间形式记录的。


## 2.3 check weather data

In [5]:
display(weather.head())
print("Weather shape:", weather.shape)
print("Weather columns:", weather.columns.tolist())

,id,datetime,conditions,temperature_c,perceived_temperature_c,humidity,windspeed_kmh
0,1,2011-01-01 00:00:00,clear,3.3,3.0,81.0,0.0
1,2,2011-01-01 01:00:00,clear,2.3,2.0,80.0,0.0
2,3,2011-01-01 02:00:00,clear,2.3,2.0,80.0,0.0
3,4,2011-01-01 03:00:00,clear,3.3,3.0,75.0,0.0
4,5,2011-01-01 04:00:00,clear,3.3,3.0,75.0,0.0


Weather shape: (17379, 7)
Weather columns: ['id', 'datetime', 'conditions', 'temperature_c', 'perceived_temperature_c', 'humidity', 'windspeed_kmh']


In [10]:
weather.info()

<class 'pandas.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       17379 non-null  int64  
 1   datetime                 17379 non-null  str    
 2   conditions               17379 non-null  str    
 3   temperature_c            17379 non-null  float64
 4   perceived_temperature_c  17379 non-null  float64
 5   humidity                 17379 non-null  float64
 6   windspeed_kmh            17379 non-null  float64
dtypes: float64(4), int64(1), str(2)
memory usage: 950.5 KB


It shows that the data is hourly weather data. 这对我们非常方便，因为 bike rentals 之后也要按小时聚合。

## 2.4 check holidays data

In [6]:
display(holidays.head())
print("Holidays shape:", holidays.shape)
print("Holidays columns:", holidays.columns.tolist())

,id,date,holiday
0,1,2011-01-17,"Dr. Martin Luther King, Jr.'s Birthday"
1,2,2011-02-21,Washington's Birthday
2,3,2011-04-15,D.C. Emancipation Day (observed)
3,4,2011-05-30,Memorial Day
4,5,2011-07-04,Independence Day


Holidays shape: (21, 3)
Holidays columns: ['id', 'date', 'holiday']


In [11]:
holidays.info()

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   id       21 non-null     int64
 1   date     21 non-null     str  
 2   holiday  21 non-null     str  
dtypes: int64(1), str(2)
memory usage: 636.0 bytes


holiday 的日期列是date, 所以他不是hourly datetime而是daily date。所以之后 merge holiday 时，不应该直接用 hourly datetime，而是要先从 rental datetime 中提取 date。

## 3. prepare hourly data
### 3.1 Convert datatime

In [14]:
registered["datetime"] = pd.to_datetime(registered["datetime"])
direct["datetime"] = pd.to_datetime(direct["datetime"])
weather["datetime"] = pd.to_datetime(weather["datetime"])
holidays["date"] = pd.to_datetime(holidays["date"])

registered.info()
direct.info()
weather.info()
holidays.info()

<class 'pandas.DataFrame'>
RangeIndex: 2672662 entries, 0 to 2672661
Data columns (total 4 columns):
 #   Column       Dtype         
---  ------       -----         
 0   id           int64         
 1   datetime     datetime64[us]
 2   user_id      int64         
 3   location_id  int64         
dtypes: datetime64[us](1), int64(3)
memory usage: 81.6 MB
<class 'pandas.DataFrame'>
RangeIndex: 620017 entries, 0 to 620016
Data columns (total 4 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   id           620017 non-null  int64         
 1   datetime     620017 non-null  datetime64[us]
 2   user_id      620017 non-null  int64         
 3   location_id  620017 non-null  int64         
dtypes: datetime64[us](1), int64(3)
memory usage: 18.9 MB
<class 'pandas.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                 

## check time ranges of rental data and weather data

In [ ]:
print("Registered datetime range:")
print(registered["datetime"].min(), "to", registered["datetime"].max())

print("\nDirect datetime range:")
print(direct["datetime"].min(), "to", direct["datetime"].max())

print("\nWeather datetime range:")
print(weather["datetime"].min(), "to", weather["datetime"].max())

print("\nHoliday date range:")
print(holidays["date"].min(), "to", holidays["date"].max())

### 确认 weather 是 hourly